In [ ]:
# imports to enable the notebook to access the src folder
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

# import the necessary packages
from src.molecules import StructuralValidator
from src.pure import PropertyCalculator
from src.pure import AntoineCalculator
from src.data import load_unifac_data
from src.mixture.unifac.pyomo_vle import pyo_vle
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# step1: define the compounds
mixture = {#'ethanol': {'CH3':1, 'CH2':1, 'OH':1},
            'A': {'CH3':1, 'CH3CO':1},
            #'B': {'CH3':2, 'CH2':5},
        #    'methanol': {'CH3OH':1},
        #   #A": {'H2O':1},
           #"A": {'CH3OH':1},
        #   "B": {'CH3':2, 'CH2':2, 'CH':1, 'HCOO':1}
           #'E': {'CH2': 3, 'HCO': 1, 'CH3COO': 1}
           #'E': {'CH3':1, 'CH2':2, 'CH3CO':1, 'CH2O':1}
           #'E': {'CH3':3, 'CH':1, 'CH2COO':1}
           }
# validate the structures
validator = StructuralValidator()
validator.validate(mixture)

In [ ]:
# step2: calculate the required properties
prop_list = ['tc', 'pc', 'tb', 'ac', 'hv']
pure_calc = PropertyCalculator()
pure_props = pure_calc.calculate(mixture, prop_list)
# fix the properties of candidates
pure_props

In [ ]:
# step3: calculate the Antoine coefficients
antoine_params_ini = {}
antoine_params = {}
antoine_calc = AntoineCalculator()
for compound, props in pure_props.items():
    results = antoine_calc.calculate_antoine_parameters(mixture, critical_properties=props, method='enthalpy_vap')
    print('optimization results was', results['termination_condition'] )
    antoine_params[compound] = {}
    antoine_params_ini[compound] = {}
    # antoine_params_ini[compound]['A'] = results['A0']
    # antoine_params_ini[compound]['B'] = results['B0']
    # antoine_params_ini[compound]['C'] = results['C0']
    antoine_params[compound]['A'] = results['A']
    antoine_params[compound]['B'] = results['B']
    antoine_params[compound]['C'] = results['C']
print(antoine_params_ini)
print(antoine_params)

In [ ]:
# step4: perform the vle

# define number of data points
n_data = 100
# set composition
x1 = np.linspace(0, 1, n_data)  # Component 1: 0 to 1
x2 = 1 - x1                     # Component 2: 1 to 0 (complement)
x = np.column_stack([x1, x2])  # Combine into 2D array
# provide nitial guess for temperature
# Extract boiling points
boiling_points = [props['tb'] for props in pure_props.values()]
T_min = min(boiling_points) 
T_max = max(boiling_points)  
T0 = np.linspace(T_min, T_max, n_data)


In [ ]:
T_range = np.linspace(298.15, T_max, 100)
P_A = antoine_calc.antoine_correlation(A=antoine_params['A']['A'],
                                         B=antoine_params['A']['B'],
                                         C=antoine_params['A']['C'],
                                         T=T_range)
P_B = antoine_calc.antoine_correlation(A=antoine_params['B']['A'],
                                         B=antoine_params['B']['B'],
                                         C=antoine_params['B']['C'],
                                         T=T_range)

# plot the results
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
#plt.plot(P_ac,T_range,  label='Acentric Factor Method')
plt.plot(P_A,T_range, label='A')
plt.plot(P_B,T_range, label='B')
# hilight Tmax, Tmin and 298.15K
plt.axhline(y=298.15, color='b', linestyle='--', label='298.15 K')
plt.axhline(y=T_max, color='r', linestyle='--', label='T_max')
plt.axhline(y=T_min, color='g', linestyle='--', label='T_min')
# add labels and legend
plt.xlabel('Saturation Pressure (bar)')
plt.ylabel('Temperature (K)')
plt.title('Saturation Pressure vs Temperature')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# step5: lets plot the results in T-xy diagram and inspect the gamma values
# Load all UNIFAC parameters
rk_data, qk_data, interaction_data = load_unifac_data()
# perform the VLE calculation
results, model = pyo_vle(x, T0, mixture, rk_data, qk_data, interaction_data, antoine_params, verbose=False)
# Extract results
# retrieving the solutions
gamma_pyo = np.empty_like(x)
T_pyo = np.empty_like(T0)
y_pyo = np.empty_like(x)
psat_pyo = np.empty_like(x)


for i in model.gamma.keys():
    gamma_pyo[i] = model.gamma[i].value

for i in model.T.keys():
    T_pyo[i] = model.T[i].value

for i in model.y.keys():
    y_pyo[i] = model.y[i].value

for i in model.psat.keys():
    psat_pyo[i] = model.psat[i].value

# Create T-xy plot
plt.figure(figsize=(10, 6))
#for i in range(x.shape[1]):
plt.plot(y_pyo[:, 0], T_pyo,  label=f'Component 1')
plt.plot(x[:, 0], T_pyo,  label=f'Component 1')
plt.ylabel('Temperature (K)')
plt.xlabel('Liquid Composition')
plt.title('T-xy Diagram')
plt.legend()
plt.grid()
plt.show()

# Inspect gamma values
print("Gamma values:")
for i in range(gamma_pyo.shape[1]):
    print(f"Component {i+1}: {gamma_pyo[:, i]}")